# Conversational RAG

This notebook covers a basic conversational retrieval workflow:

- chat history + retrieval
- LangGraph-style stateful RAG
- routing
- semantic caching
- session memory

LangChain’s memory docs say short-term memory is thread-scoped and managed through LangGraph state and checkpoints, while long-term memory persists across threads and sessions via LangGraph stores. The agentic RAG and routing docs show how a graph can generate a query and how a stateless router can be wrapped as a tool inside a conversational agent. 

## Learning goals

By the end of this notebook, you should be able to:

1. Keep chat history as part of the retrieval process.
2. Route a request to retrieval, cache, or direct answer logic.
3. Store per-session conversation state.
4. Add a simple semantic cache.
5. Understand how this fits into LangGraph stateful RAG.

## 1) Install packages

```bash
pip install -U langchain langchain-community langchain-text-splitters langchain-huggingface faiss-cpu sentence-transformers
```

This notebook keeps things simple and does not require a full agent runtime.

In [ ]:
%pip install -qU langchain langchain-community langchain-text-splitters langchain-huggingface faiss-cpu sentence-transformers

## 2) Create a tiny knowledge base

We will use a few short documents so the retrieval behavior is easy to inspect.

In [ ]:
from langchain_core.documents import Document

knowledge_docs = [
    Document(
        page_content="Short-term memory in LangGraph is thread-scoped and stores conversation history within a single session.",
        metadata={"source": "memory_doc", "topic": "short_term_memory"},
    ),
    Document(
        page_content="Long-term memory persists across threads and sessions and is backed by LangGraph stores.",
        metadata={"source": "memory_doc", "topic": "long_term_memory"},
    ),
    Document(
        page_content="A router can send a request to retrieval, cache lookup, or direct answer logic depending on intent.",
        metadata={"source": "router_doc", "topic": "routing"},
    ),
    Document(
        page_content="Hybrid or conversational RAG often combines chat history with retrieval so follow-up questions can use context.",
        metadata={"source": "rag_doc", "topic": "conversational_rag"},
    ),
]

knowledge_docs

## 3) Split and embed the documents

We will use a local CPU embedding model and a small FAISS index for retrieval.

LangChain’s memory and context docs describe state as the short-term memory for a run, and the retrieval docs show the same general pattern of splitting documents, embedding them, and searching the resulting vector store. 

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
chunks = splitter.split_documents(knowledge_docs)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embeddings)

retriever = vector_store.as_retriever(search_kwargs={"k": 2})

print("Chunks:", len(chunks))

## 4) Basic chat history + retrieval

In conversational RAG, the current user question should be interpreted together with the chat history.

A simple way to do this is:
- keep a list of messages
- rewrite or enrich the latest question using history
- retrieve relevant context
- answer using both the history and retrieved context

In [ ]:
def get_last_user_message(messages):
    for msg in reversed(messages):
        if msg["role"] == "user":
            return msg["content"]
    return ""

def build_retrieval_query(messages):
    last_user = get_last_user_message(messages)
    history_snippet = []
    for msg in messages[-4:]:
        history_snippet.append(f"{msg['role']}: {msg['content']}")
    return last_user + "Conversation context:" + "".join(history_snippet)

messages = [
    {"role": "user", "content": "What is short-term memory?"},
    {"role": "assistant", "content": "It stores conversation state within a thread."},
    {"role": "user", "content": "How is that different from long-term memory?"},
]

retrieval_query = build_retrieval_query(messages)
print(retrieval_query)

results = retriever.invoke(retrieval_query)
for i, doc in enumerate(results, 1):
    print(f"""
---Retrieved {i} ---""")
    print(doc.page_content)
    print(doc.metadata)

## 5) LangGraph-style stateful RAG

LangGraph treats the state object as short-term memory during a run, and the agentic RAG docs show graph steps operating on a `MessagesState` that includes a `messages` key. This notebook uses a simple Python dictionary to mirror that idea without building a full graph. 

In [ ]:
state = {
    "messages": [],
    "retrieved_docs": [],
    "route": None,
    "session_id": "session-001",
}

def update_state(state, role, content):
    state["messages"].append({"role": role, "content": content})

update_state(state, "user", "How is short-term memory different from long-term memory?")
state["retrieved_docs"] = retriever.invoke(build_retrieval_query(state["messages"]))
state["route"] = "retrieval"

state

## 6) Routing

A router chooses where the request should go.

For a basic conversational RAG system, useful routes are:

- `cache`
- `retrieval`
- `direct`

LangChain’s router docs show a stateless router wrapped as a tool so a conversational agent can handle memory and context while the router focuses on routing.

In [ ]:
def route_query(question: str) -> str:
    q = question.lower()

    if any(word in q for word in ["again", "same", "repeat", "cached"]):
        return "cache"

    if any(word in q for word in ["memory", "rag", "retrieval", "context", "session"]):
        return "retrieval"

    return "direct"

tests = [
    "Tell me about memory again",
    "How does retrieval work in RAG?",
    "Say hello",
]

for q in tests:
    print(q, "->", route_query(q))

## 7) Session memory

Short-term memory is scoped to a single thread or conversation, while long-term memory persists across sessions. In this notebook, we model session memory with a simple in-memory store keyed by `session_id`. 

In [ ]:
from collections import defaultdict

session_memory = defaultdict(list)

def append_session_message(session_id: str, role: str, content: str):
    session_memory[session_id].append({"role": role, "content": content})

def get_session_history(session_id: str):
    return session_memory[session_id]

append_session_message("session-001", "user", "What is short-term memory?")
append_session_message("session-001", "assistant", "It is thread-scoped memory inside LangGraph state.")
append_session_message("session-001", "user", "And long-term memory?")

get_session_history("session-001")

## 8) Simple semantic caching

Semantic caching means reusing an answer when a new question is close in meaning to a previously answered question.

LangSmith’s caching docs describe server-side caching for Agent Server deployments, and the prompt-management docs mention in-memory prompt caching for pulled prompts. This notebook uses a very small local semantic cache so the idea stays easy to understand. 

In [ ]:
import numpy as np

cache_questions = []
cache_answers = []
cache_vectors = []

def embed_text(text: str):
    return np.array(embeddings.embed_query(text), dtype=float)

def cosine_sim(a, b):
    denom = (np.linalg.norm(a) * np.linalg.norm(b))
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)

def cache_lookup(question: str, threshold: float = 0.85):
    if not cache_questions:
        return None, 0.0

    q_vec = embed_text(question)
    best_idx = -1
    best_score = -1.0

    for i, cached_vec in enumerate(cache_vectors):
        score = cosine_sim(q_vec, cached_vec)
        if score > best_score:
            best_score = score
            best_idx = i

    if best_score >= threshold:
        return cache_answers[best_idx], best_score

    return None, best_score

def cache_store(question: str, answer: str):
    cache_questions.append(question)
    cache_answers.append(answer)
    cache_vectors.append(embed_text(question))

cache_store("What is short-term memory?", "Short-term memory keeps thread-scoped conversation state.")
cached_answer, score = cache_lookup("Explain short-term memory")
print("Cached answer:", cached_answer)
print("Similarity score:", score)

## 9) A basic conversational RAG flow

The flow is:

1. Check the cache.
2. Route the request.
3. If needed, build a retrieval query from chat history.
4. Retrieve documents.
5. Form the answer.
6. Save the result into session memory and cache.

In [ ]:
def conversational_rag_answer(session_id: str, user_question: str):
    append_session_message(session_id, "user", user_question)

    cached_answer, score = cache_lookup(user_question)
    if cached_answer:
        route = "cache"
        answer = cached_answer
        retrieved = []
    else:
        route = route_query(user_question)

        if route == "retrieval":
            history = get_session_history(session_id)
            retrieval_query = build_retrieval_query(history)
            retrieved = retriever.invoke(retrieval_query)
            context = "".join(doc.page_content for doc in retrieved)
            answer = f"Based on the retrieved context: {context[:220]}..."
        else:
            retrieved = []
            answer = "This looks like a direct conversational response."

        cache_store(user_question, answer)

    append_session_message(session_id, "assistant", answer)
    return {
        "route": route,
        "answer": answer,
        "retrieved_count": len(retrieved),
    }

result = conversational_rag_answer("session-001", "How is short-term memory different from long-term memory?")
result

## 10) Follow-up question behavior

The main value of conversational RAG is that follow-up questions can reuse the session history.

That means a follow-up like “and what about the other one?” can still be grounded in the earlier turn.

In [ ]:
follow_up = conversational_rag_answer("session-001", "And what about the other one?")
follow_up

## 11) Long-term memory idea

Long-term memory is where you would store user preferences or durable application facts across conversations.

LangChain’s long-term memory docs say it persists across threads and is built on LangGraph stores. In this notebook, we keep it conceptual and use a simple dictionary to show the pattern.

In [ ]:
long_term_memory = {}

def remember_user_profile(user_id: str, key: str, value: str):
    long_term_memory.setdefault(user_id, {})
    long_term_memory[user_id][key] = value

def recall_user_profile(user_id: str):
    return long_term_memory.get(user_id, {})

remember_user_profile("user-1", "preferred_style", "concise")
remember_user_profile("user-1", "topic_interest", "RAG")

recall_user_profile("user-1")

## 12) Tool-wrapper routing pattern

The router docs show a simple pattern where the router is wrapped as a tool and a conversational agent handles the memory and context. That separation keeps routing logic stateless and easier to test.

In [ ]:
# Conceptual tool-wrapper example

def router_tool(query: str) -> str:
    route = route_query(query)
    if route == "retrieval":
        return "Would call retrieval workflow"
    if route == "cache":
        return "Would return cached answer"
    return "Would answer directly"

for q in tests:
    print(q, "->", router_tool(q))

## 13) What LangGraph adds

LangGraph is useful when you want:
- persistent state
- multi-turn memory
- branching paths
- explicit routing
- checkpoint-based conversation continuity

The LangGraph workflow docs explain that workflows follow predetermined paths, while agents are dynamic. For conversational RAG, the graph gives you a clean place to keep state and routing logic.

## Key takeaways

- Short-term memory is thread-scoped state.
- Long-term memory persists across sessions.
- Chat history should influence retrieval for follow-up questions.
- Routing can decide whether to use cache, retrieval, or direct answer logic.
- Semantic caching reduces repeated work.
- LangGraph gives you a natural place to store and update conversational state. 

## References

- Short-term memory: https://docs.langchain.com/oss/python/langchain/short-term-memory
- Long-term memory: https://docs.langchain.com/oss/python/langchain/long-term-memory
- Agentic RAG: https://docs.langchain.com/oss/python/langgraph/agentic-rag
- Router tool wrapper: https://docs.langchain.com/oss/python/langchain/multi-agent/router
- LangSmith caching: https://docs.langchain.com/langsmith/caching
- Prompt caching: https://docs.langchain.com/langsmith/manage-prompts-programmatically